# Chapter 3 — Interpolation and Polynomial Approximation

## Lagrange Interpolating Polynomial

Given $n+1$ distinct points $(x_0, y_0), (x_1, y_1), \dots, (x_n, y_n)$, the **Lagrange interpolating polynomial** is the unique polynomial of degree $\le n$ that passes through all the points:

$$P(x) = \sum_{k=0}^{n} y_k \, L_k(x)$$

where the **Lagrange basis polynomials** are

$$L_k(x) = \prod_{\substack{j=0 \\ j \ne k}}^{n} \frac{x - x_j}{x_k - x_j}$$

Each $L_k$ satisfies $L_k(x_k) = 1$ and $L_k(x_j) = 0$ for $j \ne k$.

In [3]:
import numpy as np
import pandas as pd

In [4]:
def lagrange_interpolation(xs, ys, x):
    """
    Evaluate the Lagrange interpolating polynomial at x.

    Parameters
    ----------
    xs : array-like of float, length n+1
        The distinct interpolation nodes x_0, ..., x_n.
    ys : array-like of float, length n+1
        The function values y_0, ..., y_n at the nodes.
    x  : float
        The point at which to estimate f(x).

    Returns
    -------
    float
        P(x) — the Lagrange interpolation estimate.
    """
    xs = np.asarray(xs, dtype=float)
    ys = np.asarray(ys, dtype=float)
    n = len(xs)

    if len(ys) != n:
        raise ValueError("xs and ys must have the same length")
    if n != len(set(xs)):
        raise ValueError("Interpolation nodes must be distinct")

    result = 0.0
    for k in range(n):
        # Build L_k(x)
        L_k = 1.0
        for j in range(n):
            if j != k:
                L_k *= (x - xs[j]) / (xs[k] - xs[j])
        result += ys[k] * L_k

    return result

### Example — Estimate $\cos(0.5)$ using 4 equally spaced nodes on $[0, 1]$

In [5]:
xs = np.linspace(0, 1, 4)          # nodes: 0, 1/3, 2/3, 1
ys = np.cos(xs)                     # f(x) = cos(x) at the nodes

x_eval = 0.5
p_approx = lagrange_interpolation(xs, ys, x_eval)
exact = np.cos(x_eval)

print(f"P({x_eval})  = {p_approx:.12f}")
print(f"cos({x_eval}) = {exact:.12f}")
print(f"Error    = {abs(p_approx - exact):.2e}")

P(0.5)  = 0.877330972372
cos(0.5) = 0.877582561890
Error    = 2.52e-04


## Neville's Method

Neville's method builds the same interpolating polynomial value as Lagrange, but uses a **recursive** tableau.

Define $Q_{i,j}$ as the value at $x$ of the interpolating polynomial through nodes $x_{i-j}, x_{i-j+1}, \dots, x_i$:

$$Q_{i,0} = y_i$$

$$Q_{i,j} = \frac{(x - x_{i-j})\,Q_{i,j-1} \;-\; (x - x_i)\,Q_{i-1,j-1}}{x_i - x_{i-j}}$$

The final entry $Q_{n,n}$ equals $P_n(x)$, the degree-$n$ interpolation at $x$.

In [6]:
def neville(xs, ys, x):
    """
    Neville's method for polynomial interpolation.

    Parameters
    ----------
    xs : array-like of float, length n+1
        Distinct interpolation nodes.
    ys : array-like of float, length n+1
        Function values at the nodes.
    x  : float
        The point at which to evaluate the interpolant.

    Returns
    -------
    Q : 2-D numpy array
        The Neville tableau Q[i, j].  Q[n, n] is the final estimate.
    """
    xs = np.asarray(xs, dtype=float)
    ys = np.asarray(ys, dtype=float)
    n = len(xs)
    Q = np.zeros((n, n))
    Q[:, 0] = ys

    for j in range(1, n):
        for i in range(j, n):
            Q[i, j] = ((x - xs[i - j]) * Q[i, j - 1]
                        - (x - xs[i]) * Q[i - 1, j - 1]) / (xs[i] - xs[i - j])

    return Q

### Example — Neville tableau for $f(x) = \cos(x)$, estimate at $x = 0.5$

Using 6 equally spaced nodes on $[0, 1]$ ($n = 5$).

In [7]:
xs = np.linspace(0, 1, 6)       # 6 nodes → degree up to 5
ys = np.cos(xs)
x_eval = 0.5

Q = neville(xs, ys, x_eval)

# Build a readable table from the tableau
columns = {f"Q(i,{j})" if j > 0 else "Q(i,0) = y_i": [] for j in range(6)}
col_names = list(columns.keys())

for i in range(6):
    for j in range(6):
        col_names_j = col_names[j]
        if j <= i:
            columns[col_names_j].append(f"{Q[i, j]:.10f}")
        else:
            columns[col_names_j].append("")

df_neville = pd.DataFrame(columns, index=[f"i={i}" for i in range(6)])
df_neville.insert(0, "x_i", [f"{v:.4f}" for v in xs])

print(f"Exact cos(0.5) = {np.cos(0.5):.12f}")
print(f"Q(5,5)         = {Q[5, 5]:.12f}")
print(f"Error           = {abs(Q[5, 5] - np.cos(0.5)):.2e}\n")
df_neville

Exact cos(0.5) = 0.877582561890
Q(5,5)         = 0.877582289355
Error           = 2.73e-07



,x_i,"Q(i,0) = y_i","Q(i,1)","Q(i,2)","Q(i,3)","Q(i,4)","Q(i,5)"
i=0,0.0000,1.0000000000,,,,,
i=1,0.2000,0.9800665778,0.9501664446,,,,
i=2,0.4000,0.9210609940,0.8915582021,0.8769061415,,,
i=3,0.6000,0.8253356149,0.8731983045,0.8777882789,0.8776412560,,
i=4,0.8000,0.6967067093,0.8896500677,0.8773112453,0.8775497621,0.8775840723,
i=5,1.0000,0.5403023059,0.9313133146,0.8792342560,0.8776317470,0.8775805064,0.8775822894


## Newton's Divided Differences

The **Newton forward divided-difference** interpolating polynomial is

$$P_n(x) = f[x_0] + \sum_{k=1}^{n} f[x_0, x_1, \dots, x_k]\,\prod_{j=0}^{k-1}(x - x_j)$$

where the divided differences are defined recursively:

$$f[x_i] = f(x_i)$$
$$f[x_i, \dots, x_{i+k}] = \frac{f[x_{i+1}, \dots, x_{i+k}] - f[x_i, \dots, x_{i+k-1}]}{x_{i+k} - x_i}$$

**Key advantage:** adding a new node only requires computing one new diagonal entry — no need to recompute from scratch.

In [8]:
def divided_differences(xs, ys):
    """
    Compute the divided-difference tableau.

    Returns
    -------
    F : 2-D numpy array
        F[i, j] = f[x_i, ..., x_{i+j}].  The top-row diagonal
        F[0, 0], F[0, 1], ..., F[0, n] gives the Newton coefficients.
    """
    xs = np.asarray(xs, dtype=float)
    ys = np.asarray(ys, dtype=float)
    n = len(xs)
    F = np.zeros((n, n))
    F[:, 0] = ys

    for j in range(1, n):
        for i in range(n - j):
            F[i, j] = (F[i + 1, j - 1] - F[i, j - 1]) / (xs[i + j] - xs[i])

    return F


def newton_poly_eval(xs, coeffs, x):
    """
    Evaluate Newton's interpolating polynomial at x using
    Horner-like nested multiplication.

    Parameters
    ----------
    xs     : nodes x_0, ..., x_n
    coeffs : Newton coefficients f[x0], f[x0,x1], ..., f[x0,...,xn]
    x      : evaluation point
    """
    n = len(coeffs) - 1
    result = coeffs[n]
    for k in range(n - 1, -1, -1):
        result = result * (x - xs[k]) + coeffs[k]
    return result

### Example — Estimate $\cos(0.5)$ with $x_0 = 0$, $h = 0.3$

Add nodes one at a time until $|P_n(0.5) - P_{n-1}(0.5)| < 10^{-8}$.

In [9]:
x0, h, x_eval, tol = 0.0, 0.3, 0.5, 1e-8
max_nodes = 20

rows = []
prev_P = None

for n in range(1, max_nodes + 1):
    xs = np.array([x0 + i * h for i in range(n)])
    ys = np.cos(xs)
    F = divided_differences(xs, ys)
    coeffs = F[0, :n]                            # top-row entries
    P = newton_poly_eval(xs, coeffs, x_eval)

    est_err = abs(P - prev_P) if prev_P is not None else np.nan
    true_err = abs(P - np.cos(x_eval))
    rows.append({
        "n (degree)": n - 1,
        "nodes": f"x0..x{n-1}",
        f"P_n({x_eval})": f"{P:.12f}",
        "|P_n - P_{n-1}|": f"{est_err:.2e}" if not np.isnan(est_err) else "—",
        "true |error|": f"{true_err:.2e}",
    })

    if prev_P is not None and est_err < tol:
        break
    prev_P = P

df_dd = pd.DataFrame(rows)
exact = np.cos(x_eval)
print(f"Converged at degree {rows[-1]['n (degree)']} with {len(rows)} nodes")
print(f"P({x_eval})   = {P:.12f}")
print(f"cos({x_eval}) = {exact:.12f}")
print(f"True error = {abs(P - exact):.2e}\n")
df_dd

Converged at degree 10 with 11 nodes
P(0.5)   = 0.877582563346
cos(0.5) = 0.877582561890
True error = 1.46e-09



,n (degree),nodes,P_n(0.5),|P_n - P_{n-1}|,true |error|
0,0,x0..x0,1.000000000000,—,1.22e-01
1,1,x0..x1,0.925560815209,7.44e-02,4.80e-02
2,2,x0..x2,0.878151168908,4.74e-02,5.69e-04
3,3,x0..x3,0.877434342309,7.17e-04,1.48e-04
4,4,x0..x4,0.877569848777,1.36e-04,1.27e-05
5,5,x0..x5,0.877585458023,1.56e-05,2.90e-06
6,6,x0..x6,0.877583094474,2.36e-06,5.33e-07
7,7,x0..x7,0.877582484245,6.10e-07,7.76e-08
8,8,x0..x8,0.877582535037,5.08e-08,2.69e-08
9,9,x0..x9,0.877582563803,2.88e-08,1.91e-09


## Hermite Interpolation via Divided Differences

The **Hermite interpolating polynomial** $H_{2n+1}(x)$ matches both $f(x_i)$ **and** $f'(x_i)$ at each node.

To build it with divided differences, create a **doubled node list** $z_0, z_1, z_2, z_3, \dots$ where each $x_i$ appears twice:

$$z_{2i} = z_{2i+1} = x_i$$

The first divided differences for repeated nodes use the derivative:

$$f[z_{2i}, z_{2i+1}] = f'(x_i)$$

All higher-order divided differences follow the standard formula. The result is a Newton-form polynomial of degree $\le 2n+1$.

In [10]:
def hermite_divided_differences(xs, ys, dys):
    """
    Build the Hermite divided-difference tableau using doubled nodes.

    Parameters
    ----------
    xs  : array-like, length m   — distinct nodes x_0, ..., x_{m-1}
    ys  : array-like, length m   — f(x_i)
    dys : array-like, length m   — f'(x_i)

    Returns
    -------
    z      : 1-D array, length 2m — the doubled node sequence
    coeffs : 1-D array, length 2m — top-row Newton coefficients
    Q      : 2-D array (2m × 2m) — full tableau
    """
    xs  = np.asarray(xs, dtype=float)
    ys  = np.asarray(ys, dtype=float)
    dys = np.asarray(dys, dtype=float)
    m = len(xs)
    N = 2 * m                       # total doubled nodes

    z = np.zeros(N)
    Q = np.zeros((N, N))

    for i in range(m):
        z[2 * i]     = xs[i]
        z[2 * i + 1] = xs[i]
        Q[2 * i, 0]     = ys[i]
        Q[2 * i + 1, 0] = ys[i]
        # first divided difference for the repeated node
        Q[2 * i + 1, 1] = dys[i]
        if i > 0:
            Q[2 * i, 1] = (Q[2 * i, 0] - Q[2 * i - 1, 0]) / (z[2 * i] - z[2 * i - 1])

    for j in range(2, N):
        for i in range(j, N):
            Q[i, j] = (Q[i, j - 1] - Q[i - 1, j - 1]) / (z[i] - z[i - j])

    coeffs = np.array([Q[i, i] for i in range(N)])   # diagonal = Newton coefficients
    return z, coeffs, Q

### Example — Estimate $\cos(0.5)$ with Hermite divided differences

$x_0 = 0$, $h = 0.3$, using $f(x) = \cos(x)$ and $f'(x) = -\sin(x)$.

Add nodes one at a time until $|H_{2n+1}(0.5) - H_{2(n-1)+1}(0.5)| < 10^{-8}$.

In [11]:
x0, h, x_eval, tol = 0.0, 0.3, 0.5, 1e-8
max_nodes = 20

rows_h = []
prev_H = None

for m in range(1, max_nodes + 1):
    xs  = np.array([x0 + i * h for i in range(m)])
    ys  = np.cos(xs)
    dys = -np.sin(xs)

    z, coeffs, Q_h = hermite_divided_differences(xs, ys, dys)
    H = newton_poly_eval(z, coeffs, x_eval)

    est_err  = abs(H - prev_H) if prev_H is not None else np.nan
    true_err = abs(H - np.cos(x_eval))
    rows_h.append({
        "m (nodes)": m,
        "poly degree": 2 * m - 1,
        f"H({x_eval})": f"{H:.12f}",
        "|H_new - H_prev|": f"{est_err:.2e}" if not np.isnan(est_err) else "—",
        "true |error|": f"{true_err:.2e}",
    })

    if prev_H is not None and est_err < tol:
        break
    prev_H = H

df_hermite = pd.DataFrame(rows_h)
exact = np.cos(x_eval)
print(f"Converged with {rows_h[-1]['m (nodes)']} nodes (degree {rows_h[-1]['poly degree']})")
print(f"H({x_eval})   = {H:.12f}")
print(f"cos({x_eval}) = {exact:.12f}")
print(f"True error = {abs(H - exact):.2e}\n")
df_hermite

Converged with 5 nodes (degree 9)
H(0.5)   = 0.877582561892
cos(0.5) = 0.877582561890
True error = 1.78e-12



,m (nodes),poly degree,H(0.5),|H_new - H_prev|,true |error|
0,1,1,1.000000000000,—,1.22e-01
1,2,3,0.877177210072,1.23e-01,4.05e-04
2,3,5,0.877582692886,4.05e-04,1.31e-07
3,4,7,0.877582561536,1.31e-07,3.55e-10
4,5,9,0.877582561892,3.56e-10,1.78e-12
